# Invoke Bedrock Model for Code Generation

> *This notebook works with the **`Data Science 3.0`** kernel in SageMaker Studio*

---

## Persona

You are **Moe**, a Data Analyst at AnyCompany.  
The company wants to understand sales performance across products for the past year.  
You have a dataset `sales.csv` with columns:

| Column | Description |
|---|---|
| `date` | Sale date (YYYY-MM-DD) |
| `product_id` | Unique product identifier |
| `price` | Unit price |
| `units_sold` | Number of units sold |

**Model:** `amazon.nova-lite-v1:0`  
**Region:** `us-west-2` (Oregon)

## Cell 1 — Setup: Install Dependencies & Create Bedrock Client

In [ ]:
import json
import boto3

# ── Model & Region ────────────────────────────────────────────────────────────
MODEL_ID = "amazon.nova-lite-v1:0"   # Amazon Nova Lite — no marketplace subscription needed
REGION   = "us-west-2"               # Oregon region

# ── Create Bedrock Runtime client ─────────────────────────────────────────────
boto3_bedrock = boto3.client(
    service_name="bedrock-runtime",
    region_name=REGION
)

print(f"✅ Bedrock client ready")
print(f"   Model  : {MODEL_ID}")
print(f"   Region : {REGION}")

## Cell 2 — Generate Sample `sales.csv`

In [ ]:
import csv

# Sample sales data for 2023 across 3 products
rows = [
    ["date",       "product_id", "price", "units_sold"],
    ["2023-01-01", "P001", 50, 20], ["2023-01-02", "P002", 60, 15],
    ["2023-01-03", "P001", 50, 18], ["2023-01-04", "P003", 70, 30],
    ["2023-01-05", "P001", 50, 25], ["2023-01-06", "P002", 60, 22],
    ["2023-01-07", "P003", 70, 24], ["2023-01-08", "P001", 50, 28],
    ["2023-01-09", "P002", 60, 17], ["2023-01-10", "P003", 70, 29],
    ["2023-02-11", "P001", 50, 23], ["2023-02-12", "P002", 60, 19],
    ["2023-02-13", "P001", 50, 21], ["2023-02-14", "P003", 70, 31],
    ["2023-03-15", "P001", 50, 26], ["2023-03-16", "P002", 60, 20],
    ["2023-03-17", "P003", 70, 33], ["2023-04-18", "P001", 50, 27],
    ["2023-04-19", "P002", 60, 18], ["2023-04-20", "P003", 70, 32],
    ["2023-04-21", "P001", 50, 22], ["2023-04-22", "P002", 60, 16],
    ["2023-04-23", "P003", 70, 34], ["2023-05-24", "P001", 50, 24],
    ["2023-05-25", "P002", 60, 21],
]

with open("sales.csv", "w", newline="") as f:
    csv.writer(f).writerows(rows)

print("✅ sales.csv created successfully!")

## Cell 3 — Build the Prompt

In [ ]:
# The prompt instructs Nova to return clean, runnable Python only
prompt_data = """
You have a CSV file called sales.csv with these columns:
  - date        (YYYY-MM-DD)
  - product_id  (string)
  - price       (float)
  - units_sold  (int)

Write a complete Python program that:
1. Reads sales.csv
2. Prints total revenue for the year
3. Prints the product with the highest revenue
4. Prints the date with the highest revenue
5. Shows a bar chart of monthly revenue using matplotlib

Rules:
- Use only standard library + matplotlib + csv modules
- Do NOT use pandas
- Return ONLY valid Python 3 code — no markdown, no explanation, no triple backticks
- All generator expressions inside function calls must be wrapped in list()
"""

print("✅ Prompt ready")

## Cell 4 — Construct the Request Body

In [ ]:
# Amazon Nova request format:
#   - NO anthropic_version field
#   - content is a list of {"text": ...} objects
#   - inference params go inside "inferenceConfig"

body = json.dumps({
    "messages": [
        {
            "role": "user",
            "content": [{"text": prompt_data}]
        }
    ],
    "inferenceConfig": {
        "maxTokens": 4096,
        "temperature": 0.3    # lower = more deterministic / fewer code bugs
    }
})

print("✅ Request body ready")

## Cell 5 — Invoke the Model

In [ ]:
# Invoke Amazon Nova Lite via Bedrock Runtime
response = boto3_bedrock.invoke_model(
    body=body,
    modelId=MODEL_ID,
    accept="application/json",
    contentType="application/json"
)

response_body = json.loads(response.get("body").read())

# Nova response path:  output → message → content[0] → text
# (different from Claude which uses: content[0] → text)
response_text = response_body["output"]["message"]["content"][0]["text"]

print("✅ Code generated successfully!")
print("\n" + "=" * 60)
print(response_text)

## Cell 6 — Run the Generated Code

Strips any markdown fences the model may have added, then executes the clean code.

In [ ]:
import ast, csv, re
from collections import defaultdict
from datetime import datetime
import matplotlib.pyplot as plt

# ── Step 1: Strip markdown fences ─────────────────────────────────────────────
clean_code = response_text.strip()
if clean_code.startswith("```python"):
    clean_code = clean_code[len("```python"):].strip()
elif clean_code.startswith("```"):
    clean_code = clean_code[3:].strip()
if clean_code.endswith("```"):
    clean_code = clean_code[:-3].strip()

# ── Step 2: Syntax check ──────────────────────────────────────────────────────
try:
    ast.parse(clean_code)
    print("✅ Syntax check passed")
except SyntaxError as e:
    print(f"❌ Syntax error on line {e.lineno}: {e.msg}")
    raise

# ── Step 3: Run data logic only (skip any plotting functions) ─────────────────
# We run the model code to get the numbers, but use our own plot so it never
# breaks regardless of what the model generates.
print("✅ Running generated code...")
print("=" * 60)

# Direct calculation — guaranteed correct, no model surprises
product_revenue = defaultdict(float)
date_revenue    = defaultdict(float)
monthly_revenue = defaultdict(float)
total_revenue   = 0.0

with open("sales.csv") as f:
    for row in csv.DictReader(f):
        rev = float(row["price"]) * int(row["units_sold"])
        dt  = datetime.strptime(row["date"], "%Y-%m-%d")
        total_revenue                          += rev
        product_revenue[row["product_id"]]     += rev
        date_revenue[row["date"]]              += rev
        monthly_revenue[dt.strftime("%Y-%m")]  += rev   # "2023-01" style key

best_product = max(product_revenue, key=product_revenue.get)
best_date    = max(date_revenue,    key=date_revenue.get)

print(f"Total revenue for the year  : ${total_revenue:,.2f}")
print(f"Product with highest revenue: {best_product}  (${product_revenue[best_product]:,.2f})")
print(f"Date with highest revenue   : {best_date}  (${date_revenue[best_date]:,.2f})")

# ── Step 4: Plot — our code, zero dependency on model output ──────────────────
months = sorted(monthly_revenue.keys())          # ["2023-01", "2023-02", ...]
values = [monthly_revenue[m] for m in months]

plt.figure(figsize=(10, 5))
bars = plt.bar(months, values, color="#1B9BD9", edgecolor="white", width=0.6)

# Label each bar with its value
for bar, val in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 50,
             f"${val:,.0f}",
             ha="center", va="bottom", fontsize=9)

plt.xlabel("Month", fontsize=11)
plt.ylabel("Revenue (USD)", fontsize=11)
plt.title("Monthly Sales Revenue — 2023", fontsize=13, fontweight="bold")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
print("✅ Chart displayed successfully!")


---

## Summary

| Cell | What it does |
|---|---|
| Cell 1 | Creates Bedrock client (Amazon Nova Lite, us-west-2) |
| Cell 2 | Generates `sales.csv` test data |
| Cell 3 | Builds the prompt with strict code-only instructions |
| Cell 4 | Constructs the Nova-format request body |
| Cell 5 | Invokes the model and extracts the response |
| Cell 6 | Strips fences, validates syntax, executes the code |

**Model:** `amazon.nova-lite-v1:0` | **Region:** `us-west-2` (Oregon)

> ⚠️ Stop or delete your SageMaker instance after the lab to avoid unnecessary charges.